**🔗 Live Deployment Link:** https://YOUR-APP-NAME.streamlit.app  *(<- apna Streamlit link yahan paste karein)*

# Assignment 3 — Applying, Evaluating & Comparing Multiple ML Algorithms
**Course:** Machine Learning Fundamentals (AIC354) | **Instructor:** Dr. Rao Muhammad Adeel Nawab
**Student:** _Name / Reg. No._

## Problem
**House Price Prediction (Regression)** — predict the median house value of a California district from its demographic and geographic features (similar in nature to the Car Price Prediction System).

## Dataset
California Housing dataset (20,640 rows, 8 features, 1 target `MedHouseVal`, in units of $100,000). The **full dataset** is used (no sampling).

## Algorithms compared (none used in the Car Price notebooks)
1. Ridge Regression (regularised linear model)
2. Support Vector Regression (SVR, RBF kernel)
3. Neural Network (MLP Regressor)
4. AdaBoost Regressor
5. Extra Trees Regressor

## Evaluation
10-Fold Cross-Validation, metric = **RMSE** (also MAE and R² for reference).

In [ ]:
import time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib
from tqdm.auto import tqdm
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import ExtraTreesRegressor, AdaBoostRegressor
warnings.filterwarnings('ignore')
print('Libraries imported')

## 1. Load the FULL dataset

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame
df.to_csv('california_housing.csv', index=False)   # saved copy of the full dataset
print('Shape:', df.shape)
df.head()

## 2. Explore the data

In [ ]:
print(df.isnull().sum().to_string())
df.describe().T

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df['MedHouseVal'].hist(bins=50, ax=ax[0], color='steelblue')
ax[0].set_title('Distribution of house value (x $100k)')
corr = df.corr()['MedHouseVal'].drop('MedHouseVal').sort_values()
corr.plot.barh(ax=ax[1], color='darkorange'); ax[1].set_title('Correlation with house value')
plt.tight_layout(); plt.show()

## 3. Features and target

In [ ]:
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']
print(X.shape, y.shape)

## 4. Define the 5 algorithms
Each model is wrapped in a Pipeline with `StandardScaler`, so scaling is fitted inside each fold only (no data leakage).

In [ ]:
models = {
    'Ridge Regression':        Ridge(alpha=1.0),
    'SVR (RBF)':               SVR(kernel='rbf', C=3.0),
    'Neural Network (MLP)':    MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, early_stopping=True, random_state=42),
    'AdaBoost':                AdaBoostRegressor(n_estimators=100, random_state=42),
    'Extra Trees':             ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}
pipes = {name: Pipeline([('scaler', StandardScaler()), ('model', m)]) for name, m in models.items()}
list(pipes)

## 5. 10-Fold Cross-Validation (with progress bar and time remaining)

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)
scoring = {'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2'}

fold_rmse, rows = {}, []
for name, pipe in tqdm(pipes.items(), desc='Evaluating models', unit='model'):
    t0 = time.time()
    res = cross_validate(pipe, X, y, cv=kf, scoring=scoring, n_jobs=-1)
    secs = time.time() - t0
    rmse = -res['test_rmse']
    fold_rmse[name] = rmse
    rows.append({'Model': name, 'RMSE (mean)': rmse.mean(), 'RMSE (std)': rmse.std(),
                 'MAE (mean)': -res['test_mae'].mean(), 'R2 (mean)': res['test_r2'].mean(),
                 'Time (s)': secs})
results = pd.DataFrame(rows).sort_values('RMSE (mean)').reset_index(drop=True)
results.index += 1

## 6. Comparison table

In [ ]:
results.style.format({'RMSE (mean)':'{:.4f}','RMSE (std)':'{:.4f}','MAE (mean)':'{:.4f}',
                      'R2 (mean)':'{:.4f}','Time (s)':'{:.1f}'}).background_gradient(subset=['RMSE (mean)'], cmap='RdYlGn_r')

## 7. Comparison charts

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
order = results['Model']
ax[0].bar(order, results['RMSE (mean)'], yerr=results['RMSE (std)'], capsize=5, color='steelblue')
ax[0].set_title('Mean RMSE (10-fold) — lower is better'); ax[0].set_ylabel('RMSE'); ax[0].tick_params(axis='x', rotation=30)
ax[1].boxplot([fold_rmse[m] for m in order]); ax[1].set_xticks(range(1, len(order)+1)); ax[1].set_xticklabels(order); ax[1].set_title('RMSE across the 10 folds'); ax[1].tick_params(axis='x', rotation=30)
ax[2].bar(order, results['Time (s)'], color='darkorange'); ax[2].set_title('Training + evaluation time (s)'); ax[2].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 8. Train the best model on the full data and save it

In [ ]:
best_name = results.loc[1, 'Model']
print('Best model:', best_name)
final_model = pipes[best_name].fit(X, y)
joblib.dump({'model': final_model, 'features': list(X.columns), 'name': best_name}, 'model.pkl')
print('Saved model.pkl')

## Deployment file: requirements.txt
This cell writes a `requirements.txt` with the exact library versions used here, so the web app loads `model.pkl` without version problems.

In [ ]:
import sklearn, numpy, pandas
with open('requirements.txt', 'w') as f:
    f.write("streamlit\n" + "scikit-learn==" + sklearn.__version__ + "\n" + "numpy==" + numpy.__version__ + "\n" + "pandas==" + pandas.__version__ + "\n" + "joblib==" + joblib.__version__ + "\n")
print(open('requirements.txt').read())

## 9. Test the saved model

In [ ]:
saved = joblib.load('model.pkl')
sample = pd.DataFrame([{'MedInc': 5.0, 'HouseAge': 20, 'AveRooms': 6.0, 'AveBedrms': 1.0,
                        'Population': 1500, 'AveOccup': 3.0, 'Latitude': 34.0, 'Longitude': -118.3}])[saved['features']]
pred = saved['model'].predict(sample)[0]
print(f'Predicted median house value: ${pred*100000:,.0f}')

In [ ]:
# Download the files (Colab only)
try:
    from google.colab import files
    files.download('model.pkl'); files.download('california_housing.csv'); files.download('requirements.txt')
except ImportError:
    print('Not running on Colab - files are in the current folder.')

## 10. Conclusion
- All five models were evaluated on the **same full dataset** with the **same 10 folds**, so the comparison is fair.
- The table above ranks them by mean RMSE. Non-linear models (SVR, neural network, Extra Trees) can capture patterns such as location effects that a linear model like Ridge cannot, so they usually obtain a lower error.
- The best model was retrained on all data, saved as `model.pkl`, and deployed as a Streamlit web app (link at the top of this notebook).